In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/atharvbhandare16/teen-mental-health-dataset-v01/Teen_Mental_Health_Dataset.csv


Reading data file into our environment 

In [3]:
df = pd.read_csv('/kaggle/input/datasets/atharvbhandare16/teen-mental-health-dataset-v01/Teen_Mental_Health_Dataset.csv')
df.head()

,User_id,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,A001,14,male,7.9,Instagram,9.9,3.9,3.01,1.5,low,2.0,2.0,1.0,0
1,A002,19,female,1.9,TikTok,8.0,3.9,3.22,0.5,high,8.0,1.0,10.0,0
2,A003,17,female,1.3,Instagram,7.3,3.7,3.92,0.0,high,2.0,9.5,2.0,0
3,A004,15,male,7.4,TikTok,3.9,1.3,3.48,0.5,medium,1.0,2.0,9.0,0
4,A005,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,8.0,5.0,2.0,0


In [4]:
import pandas as pd
import sqlite3

df = pd.read_csv('/kaggle/input/datasets/atharvbhandare16/teen-mental-health-dataset-v01/Teen_Mental_Health_Dataset.csv')

# load into in-memory SQLite database
conn = sqlite3.connect(':memory:')
df.to_sql('mental_health', conn, index=False, if_exists='replace')

print("Table loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Table loaded successfully.
Shape: (1200, 14)
Columns: ['User_id', 'age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']


In [5]:
def sql(query):
    query_stripped = query.strip().upper()
    if query_stripped.startswith("SELECT") or query_stripped.startswith("WITH"):
        return pd.read_sql_query(query, conn)
    else:
        conn.execute(query)
        conn.commit()
        print("Query executed successfully.")

Feature Engineering 

In [6]:
# create a temp table
sql("""
    create TEMP TABLE Data_prep_01 AS
    SELECT *,
    case when gender='male' then 1 else 0 end as gender_male_flag,
    case when gender='female' then 1 else 0 end as gender_female_flag,
    case when platform_usage in ('Instagram','Both') then 1 else 0 end as Instagram_user_flag,
    case when platform_usage in ('TikTok','Both') then 1 else 0 end as Tiktok_user_flag,
    CASE 
        WHEN social_interaction_level = 'low'    THEN 0
        WHEN social_interaction_level = 'medium' THEN 1
        WHEN social_interaction_level = 'high'   THEN 2
    END as social_interaction_encoded
    FROM mental_health

""")

Query executed successfully.


Data EDA 

In [7]:
import pandas as pd
import sqlite3

df = pd.read_sql_query("SELECT * FROM Data_prep_01", conn)

print("=" * 60)
print("1. SHAPE AND DATA TYPES")
print("=" * 60)
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)

print("\n" + "=" * 60)
print("2. FIRST 5 ROWS")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("3. MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]
if missing_df.empty:
    print("No missing values found.")
else:
    display(missing_df)

print("\n" + "=" * 60)
print("4. DUPLICATES")
print("=" * 60)
print("Total duplicate rows:", df.duplicated().sum())

print("\n" + "=" * 60)
print("5. TARGET VARIABLE DISTRIBUTION")
print("=" * 60)
print(df['depression_label'].value_counts())
print()
print(df['depression_label'].value_counts(normalize=True).mul(100).round(2).astype(str) + " %")

print("\n" + "=" * 60)
print("6. STATISTICAL SUMMARY")
print("=" * 60)
display(df.describe().T)

print("\n" + "=" * 60)
print("7. UNIQUE VALUES PER COLUMN")
print("=" * 60)
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique  |  sample: {df[col].unique()[:5]}")

print("\n" + "=" * 60)
print("8. CATEGORICAL COLUMNS — FULL VALUE COUNTS")
print("=" * 60)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"\n── {col} ──")
    print(df[col].value_counts())

print("\n" + "=" * 60)
print("9. NUMERIC FEATURES — AVERAGE BY TARGET LABEL")
print("=" * 60)
exclude = ['depression_label', 'gender_male_flag', 'gender_female_flag',
           'Instagram_user_flag', 'Tiktok_user_flag']
num_cols = [c for c in df.select_dtypes(include=['int64','float64']).columns if c not in exclude]
display(df.groupby('depression_label')[num_cols].mean().round(2))

print("\n" + "=" * 60)
print("10. FLAG COLUMNS — DEPRESSION RATE WITHIN EACH FLAG")
print("=" * 60)
flag_cols = ['gender_male_flag', 'Instagram_user_flag', 'Tiktok_user_flag']
for col in flag_cols:
    total     = df[col].sum()
    depressed = df[df[col] == 1]['depression_label'].sum()
    pct       = round(depressed / total * 100, 2) if total > 0 else 0
    print(f"{col}: {int(total)} teens flagged → {int(depressed)} depressed ({pct}%)")

print("\n" + "=" * 60)
print("11. CORRELATION WITH TARGET")
print("=" * 60)
corr_cols = num_cols + flag_cols
corr = df[corr_cols + ['depression_label']].corr()['depression_label'].drop('depression_label')
print(corr.sort_values(ascending=False).round(3))

print("\n" + "=" * 60)
print("12. OUTLIER CHECK (IQR METHOD)")
print("=" * 60)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers")

1. SHAPE AND DATA TYPES
Shape: (1200, 19)

Data types:
 User_id                        object
age                             int64
gender                         object
daily_social_media_hours      float64
platform_usage                 object
sleep_hours                   float64
screen_time_before_sleep      float64
academic_performance          float64
physical_activity             float64
social_interaction_level       object
stress_level                  float64
anxiety_level                 float64
addiction_level               float64
depression_label                int64
gender_male_flag                int64
gender_female_flag              int64
Instagram_user_flag             int64
Tiktok_user_flag                int64
social_interaction_encoded      int64
dtype: object

2. FIRST 5 ROWS


,User_id,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,gender_male_flag,gender_female_flag,Instagram_user_flag,Tiktok_user_flag,social_interaction_encoded
0,A001,14,male,7.9,Instagram,9.9,3.9,3.01,1.5,low,2.0,2.0,1.0,0,1,0,1,0,0
1,A002,19,female,1.9,TikTok,8.0,3.9,3.22,0.5,high,8.0,1.0,10.0,0,0,1,0,1,2
2,A003,17,female,1.3,Instagram,7.3,3.7,3.92,0.0,high,2.0,9.5,2.0,0,0,1,1,0,2
3,A004,15,male,7.4,TikTok,3.9,1.3,3.48,0.5,medium,1.0,2.0,9.0,0,1,0,0,1,1
4,A005,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,8.0,5.0,2.0,0,0,1,1,1,1



3. MISSING VALUES
No missing values found.

4. DUPLICATES
Total duplicate rows: 0

5. TARGET VARIABLE DISTRIBUTION
depression_label
0    1169
1      31
Name: count, dtype: int64

depression_label
0    97.42 %
1     2.58 %
Name: proportion, dtype: object

6. STATISTICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,1200.0,15.928333,2.021947,13.0,14.000,16.00,18.00,19.0
daily_social_media_hours,1200.0,4.590917,2.122337,1.0,2.575,4.60,6.40,8.7
sleep_hours,1200.0,5.970500,2.019045,3.0,4.300,5.30,7.90,9.9
screen_time_before_sleep,1200.0,2.074000,1.158876,0.3,1.100,1.70,3.30,3.9
academic_performance,1200.0,2.990383,0.576758,2.0,2.500,2.99,3.48,4.0
physical_activity,1200.0,0.951583,0.524954,0.0,0.600,0.90,1.40,1.9
stress_level,1200.0,5.405833,3.026414,1.0,2.000,5.00,8.00,10.0
anxiety_level,1200.0,5.653750,2.998941,1.0,2.000,6.00,8.00,10.0
addiction_level,1200.0,5.482500,3.001200,1.0,2.000,6.00,8.00,10.0
depression_label,1200.0,0.025833,0.158704,0.0,0.000,0.00,0.00,1.0



7. UNIQUE VALUES PER COLUMN
User_id: 1200 unique  |  sample: ['A001' 'A002' 'A003' 'A004' 'A005']
age: 7 unique  |  sample: [14 19 17 15 18]
gender: 2 unique  |  sample: ['male' 'female']
daily_social_media_hours: 67 unique  |  sample: [7.9 1.9 1.3 7.4 4.7]
platform_usage: 3 unique  |  sample: ['Instagram' 'TikTok' 'Both']
sleep_hours: 41 unique  |  sample: [9.9 8.  7.3 3.9 4.9]
screen_time_before_sleep: 19 unique  |  sample: [3.9 3.7 1.3 3.  0.7]
academic_performance: 201 unique  |  sample: [3.01 3.22 3.92 3.48 2.37]
physical_activity: 16 unique  |  sample: [1.5 0.5 0.  1.4 0.6]
social_interaction_level: 3 unique  |  sample: ['low' 'high' 'medium']
stress_level: 10 unique  |  sample: [2. 8. 1. 6. 5.]
anxiety_level: 10 unique  |  sample: [ 2.   1.   9.5  5.  10. ]
addiction_level: 10 unique  |  sample: [ 1. 10.  2.  9.  5.]
depression_label: 2 unique  |  sample: [0 1]
gender_male_flag: 2 unique  |  sample: [1 0]
gender_female_flag: 2 unique  |  sample: [0 1]
Instagram_user_flag: 2 uni

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,social_interaction_encoded
depression_label,,,,,,,,,,
0,15.92,4.54,6.00,2.07,2.99,0.95,5.37,5.61,5.47,0.96
1,16.06,6.39,4.87,2.24,3.00,0.92,6.74,7.45,6.10,1.03



10. FLAG COLUMNS — DEPRESSION RATE WITHIN EACH FLAG
gender_male_flag: 615 teens flagged → 14 depressed (2.28%)
Instagram_user_flag: 802 teens flagged → 19 depressed (2.37%)
Tiktok_user_flag: 789 teens flagged → 21 depressed (2.66%)

11. CORRELATION WITH TARGET
daily_social_media_hours      0.138
anxiety_level                 0.098
stress_level                  0.072
addiction_level               0.033
screen_time_before_sleep      0.023
social_interaction_encoded    0.014
age                           0.011
Tiktok_user_flag              0.007
academic_performance          0.001
physical_activity            -0.011
Instagram_user_flag          -0.019
gender_male_flag             -0.020
sleep_hours                  -0.089
Name: depression_label, dtype: float64

12. OUTLIER CHECK (IQR METHOD)
age: 0 outliers
daily_social_media_hours: 0 outliers
sleep_hours: 0 outliers
screen_time_before_sleep: 0 outliers
academic_performance: 0 outliers
physical_activity: 0 outliers
stress_level: 0 outlie

Dropping unnecessay Columns 

In [8]:
# ── drop columns that are not needed ─────────────────────────────
drop_cols = [
    'User_id',                   # serial number
    'gender',                    # already encoded
    'platform_usage',            # already encoded
    'social_interaction_level',  # already encoded
    'gender_female_flag'         # dummy variable trap
]

df_model = df.drop(columns=drop_cols)

print("Remaining columns:", df_model.columns.tolist())
print("Shape:", df_model.shape)


Remaining columns: ['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label', 'gender_male_flag', 'Instagram_user_flag', 'Tiktok_user_flag', 'social_interaction_encoded']
Shape: (1200, 14)


Defining features and target variable 

In [9]:
# ── define features and target ────────────────────────────────────
X = df_model.drop(columns=['depression_label'])
y = df_model['depression_label']

print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nPositive class count:", y.sum())
print("Negative class count:", (y == 0).sum())
print("Scale pos weight to use:", round((y == 0).sum() / y.sum(), 1))

Features: ['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level', 'gender_male_flag', 'Instagram_user_flag', 'Tiktok_user_flag', 'social_interaction_encoded']
X shape: (1200, 13)
y shape: (1200,)

Positive class count: 31
Negative class count: 1169
Scale pos weight to use: 37.7


Train test split (80-20%)

In [10]:
# ── train test split ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| positives:", y_train.sum())
print("X_test :", X_test.shape,  "| positives:", y_test.sum())

X_train: (960, 13) | positives: 25
X_test : (240, 13) | positives: 6


Training XG Boost model 

In [11]:
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# ── base xgboost model ────────────────────────────────────────────
base_model = XGBClassifier(
    n_estimators   = 100,
    learning_rate  = 0.1,
    max_depth      = 4,
    subsample      = 0.8,
    scale_pos_weight = 38,
    random_state   = 42,
    eval_metric    = 'logloss',
    verbosity      = 0
)

# ── wrap in balanced bagging classifier ───────────────────────────
bbc = BalancedBaggingClassifier(
    estimator          = base_model,
    n_estimators       = 5,        # 5 buckets
    sampling_strategy  = 0.2,      # 1:5 ratio (pos:neg)
    replacement        = True,     # with replacement (bagging)
    random_state       = 42
)

# ── stratified kfold ──────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── cross validation scores ───────────────────────────────────────
print("Running cross validation...")
print("(5 folds × 5 buckets = 25 models trained)\n")

recall_scores    = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='recall')
precision_scores = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='precision')
f1_scores        = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='f1')

print("── Recall per fold    :", recall_scores.round(3))
print("── Mean Recall        :", recall_scores.mean().round(3))
print("── Std Recall         :", recall_scores.std().round(3))

print("\n── Precision per fold :", precision_scores.round(3))
print("── Mean Precision     :", precision_scores.mean().round(3))
print("── Std Precision      :", precision_scores.std().round(3))

print("\n── F1 per fold        :", f1_scores.round(3))
print("── Mean F1            :", f1_scores.mean().round(3))
print("── Std F1             :", f1_scores.std().round(3))

Running cross validation...
(5 folds × 5 buckets = 25 models trained)

── Recall per fold    : [0.4 0.6 0.2 0.6 0.6]
── Mean Recall        : 0.48
── Std Recall         : 0.16

── Precision per fold : [0.154 0.158 0.056 0.273 0.176]
── Mean Precision     : 0.163
── Std Precision      : 0.069

── F1 per fold        : [0.222 0.25  0.087 0.375 0.273]
── Mean F1            : 0.241
── Std F1             : 0.093


In [13]:
# ── train final model on full training set ────────────────────────
print("Training final model on full X_train...")
bbc.fit(X_train, y_train)
print("Done.")

Training final model on full X_train...
Done.


Evaluation metrics

In [14]:
# ── evaluate on test set ──────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred  = bbc.predict(X_test)
y_test_proba = bbc.predict_proba(X_test)[:, 1]

print("=" * 50)
print("FINAL EVALUATION ON TEST SET")
print("=" * 50)
print(classification_report(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print(f"                   Predicted 0   Predicted 1")
print(f"Actual 0 (healthy)     {cm[0][0]}           {cm[0][1]}")
print(f"Actual 1 (depressed)   {cm[1][0]}           {cm[1][1]}")

print("\nBreakdown:")
print(f"True  Negatives  (TN) - correctly predicted healthy   : {cm[0][0]}")
print(f"False Positives  (FP) - healthy flagged as depressed  : {cm[0][1]}")
print(f"False Negatives  (FN) - depressed teens missed        : {cm[1][0]}")
print(f"True  Positives  (TP) - depressed teens caught        : {cm[1][1]}")

FINAL EVALUATION ON TEST SET
              precision    recall  f1-score   support

           0       0.99      0.93      0.96       234
           1       0.19      0.67      0.30         6

    accuracy                           0.92       240
   macro avg       0.59      0.80      0.63       240
weighted avg       0.97      0.92      0.94       240

Confusion Matrix:
                   Predicted 0   Predicted 1
Actual 0 (healthy)     217           17
Actual 1 (depressed)   2           4

Breakdown:
True  Negatives  (TN) - correctly predicted healthy   : 217
False Positives  (FP) - healthy flagged as depressed  : 17
False Negatives  (FN) - depressed teens missed        : 2
True  Positives  (TP) - depressed teens caught        : 4


In [15]:
# ── threshold analysis ────────────────────────────────────────────
import pandas as pd

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []

for t in thresholds:
    y_pred = (y_test_proba >= t).astype(int)
    TP = int(((y_pred == 1) & (y_test == 1)).sum())
    TN = int(((y_pred == 0) & (y_test == 0)).sum())
    FP = int(((y_pred == 1) & (y_test == 0)).sum())
    FN = int(((y_pred == 0) & (y_test == 1)).sum())
    precision = round(TP / (TP + FP) * 100, 1) if (TP + FP) > 0 else 0
    recall    = round(TP / (TP + FN) * 100, 1) if (TP + FN) > 0 else 0
    f1        = round(2 * precision * recall / (precision + recall), 1) if (precision + recall) > 0 else 0
    rows.append({'Threshold': t, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
                 'Precision %': precision, 'Recall %': recall, 'F1 %': f1})

display(pd.DataFrame(rows))

,Threshold,TP,TN,FP,FN,Precision %,Recall %,F1 %
0,0.1,6,154,80,0,7.0,100.0,13.1
1,0.2,6,187,47,0,11.3,100.0,20.3
2,0.3,6,199,35,0,14.6,100.0,25.5
3,0.4,5,208,26,1,16.1,83.3,27.0
4,0.5,4,217,17,2,19.0,66.7,29.6
5,0.6,4,224,10,2,28.6,66.7,40.0
6,0.7,4,228,6,2,40.0,66.7,50.0
7,0.8,2,231,3,4,40.0,33.3,36.3
8,0.9,1,233,1,5,50.0,16.7,25.0


In [16]:
# ── evaluate on train and test at multiple thresholds ──────────────
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss

y_train_proba = bbc.predict_proba(X_train)[:, 1]
y_test_proba  = bbc.predict_proba(X_test)[:, 1]

thresholds = [round(t, 1) for t in np.arange(0.1, 1.0, 0.1)]

def get_metrics_table(y_true, y_proba, dataset_name):
    rows = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        TP = int(((y_pred == 1) & (y_true == 1)).sum())
        TN = int(((y_pred == 0) & (y_true == 0)).sum())
        FP = int(((y_pred == 1) & (y_true == 0)).sum())
        FN = int(((y_pred == 0) & (y_true == 1)).sum())

        accuracy  = round((TP + TN) / (TP + TN + FP + FN) * 100, 1)
        precision = round(TP / (TP + FP) * 100, 1) if (TP + FP) > 0 else 0.0
        recall    = round(TP / (TP + FN) * 100, 1) if (TP + FN) > 0 else 0.0
        f1        = round(2 * precision * recall / (precision + recall), 1) if (precision + recall) > 0 else 0.0

        rows.append({
            'Threshold'  : t,
            'TP'         : TP,
            'TN'         : TN,
            'FP'         : FP,
            'FN'         : FN,
            'Accuracy %' : accuracy,
            'Precision %': precision,
            'Recall %'   : recall,
            'F1 %'       : f1
        })

    df_result = pd.DataFrame(rows)
    print(f"\n{'='*95}")
    print(f"{dataset_name} — THRESHOLD-WISE METRICS")
    print(f"{'='*95}")
    display(df_result)
    return df_result

train_results = get_metrics_table(y_train, y_train_proba, "TRAIN SET")
test_results  = get_metrics_table(y_test, y_test_proba, "TEST SET")


TRAIN SET — THRESHOLD-WISE METRICS


,Threshold,TP,TN,FP,FN,Accuracy %,Precision %,Recall %,F1 %
0,0.1,25,640,295,0,69.3,7.8,100.0,14.5
1,0.2,25,739,196,0,79.6,11.3,100.0,20.3
2,0.3,24,797,138,1,85.5,14.8,96.0,25.6
3,0.4,23,842,93,2,90.1,19.8,92.0,32.6
4,0.5,21,869,66,4,92.7,24.1,84.0,37.5
5,0.6,20,891,44,5,94.9,31.2,80.0,44.9
6,0.7,18,912,23,7,96.9,43.9,72.0,54.5
7,0.8,15,922,13,10,97.6,53.6,60.0,56.6
8,0.9,8,929,6,17,97.6,57.1,32.0,41.0



TEST SET — THRESHOLD-WISE METRICS


,Threshold,TP,TN,FP,FN,Accuracy %,Precision %,Recall %,F1 %
0,0.1,6,154,80,0,66.7,7.0,100.0,13.1
1,0.2,6,187,47,0,80.4,11.3,100.0,20.3
2,0.3,6,199,35,0,85.4,14.6,100.0,25.5
3,0.4,5,208,26,1,88.8,16.1,83.3,27.0
4,0.5,4,217,17,2,92.1,19.0,66.7,29.6
5,0.6,4,224,10,2,95.0,28.6,66.7,40.0
6,0.7,4,228,6,2,96.7,40.0,66.7,50.0
7,0.8,2,231,3,4,97.1,40.0,33.3,36.3
8,0.9,1,233,1,5,97.5,50.0,16.7,25.0


In [17]:
# ── log loss ────────────────────────────────────────────────────────
train_logloss = log_loss(y_train, y_train_proba)
test_logloss  = log_loss(y_test, y_test_proba)

print("=" * 50)
print("LOG LOSS")
print("=" * 50)
print(f"Train Log Loss : {train_logloss:.4f}")
print(f"Test  Log Loss : {test_logloss:.4f}")
print(f"Gap            : {abs(train_logloss - test_logloss):.4f}")
print("\n(<0.2=excellent, 0.2-0.4=good, >0.6=poor)")

LOG LOSS
Train Log Loss : 0.1832
Test  Log Loss : 0.1861
Gap            : 0.0029

(<0.2=excellent, 0.2-0.4=good, >0.6=poor)
